# 05. weight를 바꾸면 결과가 바뀔까?

            이번 노트북은 수학적 모델링에서 매우 중요한 질문을 다룹니다.

            > 우리가 정한 weight가 달라지면 결론도 달라질까?

            기본 balanced는 네 항목을 모두 0.25로 둡니다.


## 오늘 사용할 말

- graph(그래프): 점과 선으로 이루어진 연결 구조
- node(꼭짓점): 지도 위의 역 후보 지점
- edge(변): 두 지점을 연결하는 하나의 경로
- weight(가중치): 어떤 edge가 좋은지 나쁜지 판단하는 점수
- normalization(정규화): 서로 단위가 다른 값을 비교 가능한 점수로 바꾸는 일
- MST, minimum spanning tree(최소신장수형도): 모든 node를 연결하되 총 비용을 작게 만드는 기본 구조
- shortest path(최단경로): graph 안에서 두 node 사이를 가장 짧게 가는 경로
- stretch(우회율): 선택한 구조에서 얼마나 돌아가는지 나타내는 값
- t-spanner(t-스패너): 너무 많이 돌아가지 않도록 edge를 추가하는 방법


## 1. 준비하기

03번에서 계산한 edge 점수를 다시 읽습니다. 여기서부터는 같은 edge에 대해 weight만 바꿔가며 반복 계산합니다.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
for path in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (path / "analysis/student_helpers.py").is_file():
        PROJECT_ROOT = path
        break
    if (path / "student_helpers.py").is_file():
        PROJECT_ROOT = path
        break

if (PROJECT_ROOT / "analysis").is_dir():
    sys.path.insert(0, str(PROJECT_ROOT / "analysis"))
else:
    sys.path.insert(0, str(PROJECT_ROOT))

from student_helpers import *

OUT = output_dir(PROJECT_ROOT)
print("작업 폴더:", PROJECT_ROOT)
print("결과 저장 폴더:", OUT)


In [ ]:
stations, _ = load_current_data(PROJECT_ROOT)
reference_edges = read_csv(OUT / "03_edge_scores.csv")
for edge in reference_edges:
    for key in ["pair_order", "from_order", "to_order"]:
        edge[key] = int(edge[key])
    for key in numeric_edge_columns():
        if key in edge:
            edge[key] = float(edge[key])
nodes = components(station_ids(stations), reference_edges)[0]
print("분석할 edge:", len(reference_edges))
print("분석할 node:", len(nodes))


## 2. 다섯 가지 weight 세팅

`balanced`는 W1~W4를 똑같이 봅니다.

나머지는 특정 기준을 더 중요하게 보는 경우입니다.


In [ ]:
scenarios = {
    "balanced": {"distance": 0.25, "lane_capacity": 0.25, "protection_proxy": 0.25, "preliminary_slope": 0.25},
    "distance_focus": {"distance": 0.55, "lane_capacity": 0.15, "protection_proxy": 0.15, "preliminary_slope": 0.15},
    "lane_focus": {"distance": 0.20, "lane_capacity": 0.50, "protection_proxy": 0.15, "preliminary_slope": 0.15},
    "protection_focus": {"distance": 0.20, "lane_capacity": 0.15, "protection_proxy": 0.50, "preliminary_slope": 0.15},
    "slope_focus": {"distance": 0.20, "lane_capacity": 0.15, "protection_proxy": 0.15, "preliminary_slope": 0.50},
}

for name, weights in scenarios.items():
    print(name, weights, "합계=", round(sum(weights.values()), 2))
    assert abs(sum(weights.values()) - 1.0) < 1e-12


## 3. weight별로 구조 다시 만들기

각 scenario마다 edge 점수를 다시 계산하고, 세 가지 구조를 다시 만듭니다.

- MST
- T2_SPANNER
- DEGREE_LIMIT_3


In [ ]:
metrics = []
edge_rows = []
edge_counter = Counter()

for scenario, weights in scenarios.items():
    scored = score_edges(reference_edges, weights)
    scenario_structures = {
        "MST": mst(nodes, scored, "scenario_cost"),
        "T2_SPANNER": greedy_spanner(nodes, scored, 2.0),
        "DEGREE_LIMIT_3": degree_limited_kruskal(nodes, scored, 3),
    }
    for structure, selected in scenario_structures.items():
        row, _ = structure_metrics(structure, nodes, scored, selected)
        row["scenario"] = scenario
        metrics.append(row)
        for edge in selected:
            edge_counter[(structure, edge["pair_id"])] += 1
            item = dict(edge)
            item["scenario"] = scenario
            item["structure"] = structure
            edge_rows.append(item)

write_csv(OUT / "05_weight_sensitivity_metrics.csv", metrics)
write_csv(OUT / "05_weight_sensitivity_edges.csv", edge_rows)
print_table(metrics, ["scenario", "structure", "edge_count", "total_w1_distance_km", "max_distance_stretch"], limit=15)


## 4. 얼마나 비슷한 결과가 나왔는지 보기

서로 다른 weight에서 같은 edge가 반복해서 선택되면, 그 edge는 비교적 안정적인 edge라고 볼 수 있습니다.

Jaccard similarity는 두 선택 결과가 얼마나 비슷한지 보는 값입니다.


In [ ]:
stability = [{"structure": key[0], "pair_id": key[1], "selection_count": value} for key, value in sorted(edge_counter.items())]
write_csv(OUT / "05_edge_stability.csv", stability)

by_case = defaultdict(set)
for row in edge_rows:
    by_case[(row["scenario"], row["structure"])].add(row["pair_id"])

pairwise = []
keys = list(by_case)
for i, a in enumerate(keys):
    for b in keys[i + 1:]:
        inter = len(by_case[a] & by_case[b])
        union = len(by_case[a] | by_case[b])
        pairwise.append({"case_a": "|".join(a), "case_b": "|".join(b), "jaccard": inter / union if union else 1.0})

write_csv(OUT / "05_pairwise_jaccard.csv", pairwise)
summary = {"status": "PASS", "scenario_count": len(scenarios), "case_count": len(by_case), "mean_jaccard": float(np.mean([r["jaccard"] for r in pairwise]))}
write_json(OUT / "05_weight_sensitivity_summary.json", summary)
print("평균 Jaccard similarity:", round(summary["mean_jaccard"], 3))
print_table(sorted(stability, key=lambda row: row["selection_count"], reverse=True), ["structure", "pair_id", "selection_count"], limit=10)


## 5. 그림으로 보기

아래 그림은 weight 세팅이 바뀔 때 구조별 총 거리가 어떻게 달라지는지 보여줍니다.

생각해볼 질문:

- 어떤 algorithm이 weight 변화에 가장 민감한가요?
- balanced 결과와 특정 기준 focus 결과가 많이 다르다면 어떻게 해석해야 할까요?


In [ ]:
scenario_order = list(scenarios)
structures = sorted({m["structure"] for m in metrics})
fig, ax = plt.subplots(figsize=(11, 6))
for structure in structures:
    values = [next(m for m in metrics if m["scenario"] == s and m["structure"] == structure)["total_w1_distance_km"] for s in scenario_order]
    ax.plot(scenario_order, values, marker="o", label=structure)
ax.set_ylabel("Total selected W1 distance (km)")
ax.set_title("Weight sensitivity by structure")
ax.tick_params(axis="x", rotation=20)
ax.grid(alpha=0.2)
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "05_weight_sensitivity.png", dpi=180)
plt.show()
